# AI-Based Exoplanet Detection using TESS Light Curves

## 1. Import Libraries

In [1]:
import lightkurve as lk
import pandas as pd
import numpy as np

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from astropy.timeseries import BoxLeastSquares

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/lightkurve/prf/__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


## 2. Load Confirmed Planet Dataset

In [3]:
planets = pd.read_csv("data/Cp.csv", comment="#")
positive_targets = (planets["tid"].dropna().unique()[:50])

## 3. Load False Positive Dataset

In [5]:
neg = pd.read_csv("data/False_positive.csv", comment="#")
negetive_targets = (neg["tid"].dropna().unique()[:50])

## 4. Feature Extraction

In [7]:
def extract_features(target):
    try:
        search = lk.search_lightcurve(f"TIC {int(target)}", mission="TESS")

        if len(search) == 0:
            print(f"No data for {target}")
            return None

        lc_collection = search[:2].download_all()

        if lc_collection is None or len(lc_collection) == 0:
            print("Download failed")
            return None

        lc = lc_collection.stitch()

        if len(lc.time) < 100:
            print("Too few data points")
            return None

        if lc is None:
            print("Download returned None")
            return None

        lc_clean = (lc.remove_nans().remove_outliers(sigma=5).flatten(window_length=1001))

        time = lc_clean.time.value
        flux = lc_clean.flux.value

        model = BoxLeastSquares(time, flux)

        duration = np.array([0.01, 0.02, 0.04, 0.06, 0.08])
        period = np.linspace(0.5, 20, 5000)

        result = model.power(period, duration)

        best_idx = np.argmax(result.power)

        if len(result.power) == 0:
            print("No BLS result")
            return None

        result = {
            "target": target,
            "period": float(result.period[best_idx]),
            "duration": float(result.duration[best_idx]),
            "depth": float(result.depth[best_idx]),
            "power": float(result.power[best_idx]),
            "flux_std": float(np.std(lc_clean.flux))
        }

        return result

    except Exception as e:

        print("ERROR OCCURRED")
        print(type(e))
        print(e)

        return None

In [8]:
positive_data = []

for star in positive_targets:
    print("Planet:", star)
    result = extract_features(star)

    if result:
        result["label"] = 1
        positive_data.append(result)

positive_df = pd.DataFrame(positive_data)

Planet: 317060587
Planet: 366989877
Planet: 320004517
Planet: 299799658
Planet: 79748331
Planet: 158297421
Planet: 351601843
Planet: 370133522
Planet: 161032923
Planet: 360630575
Planet: 383390264
Planet: 290348383
Planet: 394561119
Planet: 254113311
Planet: 154872375
Planet: 142276270
Planet: 232967440
Planet: 154089169
Planet: 266980320
Planet: 158002130
Planet: 229510866
Planet: 233087860
Planet: 147950620
Planet: 99869022
Planet: 394137592
Planet: 29960110
Planet: 23434737
Planet: 231702397
Planet: 349095149
Planet: 299798795
Planet: 360156606
Planet: 300038935
Planet: 290131778
Planet: 287156968
Planet: 447061717
Planet: 364395234
Planet: 260647166
Planet: 103633434
Planet: 153951307
Planet: 219698776
Planet: 219850915
Planet: 230127302
Planet: 232540264
Planet: 232612416
Planet: 232976128
Planet: 52368076
Planet: 237222864
Planet: 288735205
Planet: 355867695
Planet: 467179528


In [9]:
negative_data = []

for star in negetive_targets:
    print("Non-planet:", star)
    result = extract_features(star)

    if result:
        result["label"] = 0
        negative_data.append(result)

negative_df = pd.DataFrame(negative_data)

Non-planet: 50365310
ERROR OCCURRED
<class 'requests.exceptions.ConnectionError'>
('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer'))
Non-planet: 124709665
Non-planet: 106997505
Non-planet: 238597883
Non-planet: 169904935
Non-planet: 156115721
Non-planet: 440801822
Non-planet: 139853601
Non-planet: 97700520
Non-planet: 96246348
Non-planet: 155044736
ERROR OCCURRED
<class 'lightkurve.utils.LightkurveError'>
Not recognized as a supported data product:
/Users/macbook/.lightkurve/cache/mastDownload/TESS/tess2021014023720-s0034-0000000155044736-0204-s/tess2021014023720-s0034-0000000155044736-0204-s_lc.fits
This file may be corrupt due to an interrupted download. Please remove it from your disk and try again.
Non-planet: 175310067
Non-planet: 182943944
Non-planet: 291555748
Non-planet: 140706664
Non-planet: 143994283
Non-planet: 238374636
Non-planet: 141482386
Non-planet: 297967252
Non-planet: 388624270
Non-planet: 447283466
Non-planet: 374908020
Non-planet: 4642960

## 5. Dataset Generation

In [10]:
dataset = pd.concat([positive_df, negative_df], ignore_index=True)
dataset.head()

,target,period,duration,depth,power,flux_std,label
0,317060587,9.140228,0.08,0.000277,0.000013,0.000608,1
1,366989877,17.175835,0.01,0.000507,0.000013,0.000968,1
2,320004517,17.129026,0.04,0.000408,0.000014,0.000797,1
3,299799658,17.109522,0.04,0.000686,0.000013,0.000766,1
4,79748331,6.444789,0.08,0.000346,0.000093,0.002233,1


In [ ]:
dataset.to_csv("exoplanet_dataset.csv", index=False)

## 6. Model Training

In [11]:
features = ["period", "duration", "depth", "power", "flux_std"]

X = dataset[features]
y = dataset["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = XGBClassifier(n_estimators=100, max_depth=4)

model.fit(X_train, y_train)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


## 7. Prediction

In [12]:
predictions = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, predictions))

Accuracy: 0.7


## Testing 

In [13]:
new_star = 317060587
features = extract_features(new_star)

X_new = [[features["period"], features["duration"], features["depth"], features["power"], features["flux_std"]]]
probability = model.predict_proba(X_new)[0][1]

if probability > 0.8:
    print("Likely Exoplanet Candidate")
elif probability > 0.5:
    print("Possible Exoplanet Candidate")
else:
    print("Likely False Positive")

print(f"Planet Probability = {probability:.2%}")

Possible Exoplanet Candidate
Planet Probability = 61.88%
